# Convert Retrieval Results to Format B

This notebook transforms `data_local_retrieval_results.json` into **Format B** (embedded in answers with retrieval_context) for chunk-level assertion evaluation.

Format B structure (embedded in answers.json). Note that `retrieval_context` is a **list** of documents, each containing a `regions` array of chunks:
```json
{
  "question_id": "...",
  "question_text": "...",
  "retrieval_context": [
    {
      "regions": [
        {"text": "...", "chunk_id": 0},
        ...
      ]
    }
  ]
}
```


## Section 1: Configure Notebook Paths and Dependencies

In [1]:
# Copyright (c) 2025 Microsoft Corporation.
import json
from pathlib import Path
from typing import Any, TypedDict

# Define paths (notebook is in chunk_format_conversion/, example_answers is sibling directory)
notebook_dir = Path()
input_file = (
    notebook_dir
    / ".."
    / "example_answers"
    / "vector_rag_long_context"
    / "data_global_retrieval_results.json"
)
output_file = notebook_dir / "chunks_b.json"

print(f"Input file: {input_file.resolve()}")
print(f"Output file: {output_file.resolve()}")
print(f"Input exists: {input_file.exists()}")

Input file: /Users/gaudy-microsoft/Repositories/benchmark-qed/docs/notebooks/example_answers/vector_rag_long_context/data_global_retrieval_results.json
Output file: /Users/gaudy-microsoft/Repositories/benchmark-qed/docs/notebooks/chunk_format_conversion/chunks_b.json
Input exists: True


## Section 2: Load `data_local_retrieval_results.json`

In [ ]:
try:
    with input_file.open() as f:
        source_data = json.load(f)

    print(f"✓ Successfully loaded {len(source_data)} records from {input_file.name}")
    print(f"  Type: {type(source_data)}")

except FileNotFoundError:
    print(f"✗ File not found: {input_file}")
    source_data = []

except json.JSONDecodeError as e:
    print(f"✗ JSON parsing error: {e}")
    source_data = []

✓ Successfully loaded 50 records from data_global_retrieval_results.json
  Type: <class 'list'>


## Section 3: Profile Source JSON Structure

In [3]:
if source_data:
    # Inspect first record
    first_record = source_data[0]
    print("=== FIRST RECORD STRUCTURE ===")
    print(f"Top-level keys: {list(first_record.keys())}")
    print(f"\nquestion_id: {first_record.get('question_id')}")
    print(f"text: {first_record.get('text')[:80]}...")

    print("\ncontext[0]:")
    if first_record.get("context"):
        context_item = first_record["context"][0]
        print(f"  Keys: {list(context_item.keys())}")
        print(f"  chunk_id: {context_item.get('chunk_id')}")
        print(f"  text length: {len(context_item.get('text', ''))}")

    print(f"\nTotal records: {len(source_data)}")
    print(f"Context items in first record: {len(first_record.get('context', []))}")

=== FIRST RECORD STRUCTURE ===
Top-level keys: ['question_id', 'text', 'context']

question_id: 064ad638-583e-4d7f-941e-597e2e70a9df
text: Which public health controversies shape government actions and legal investigati...

context[0]:
  Keys: ['chunk_id', 'text']
  chunk_id: 5493
  text length: 1004

Total records: 50
Context items in first record: 200


## Section 4: Define Option B Target Schema

**Option B Format (Embedded in answers):**
- `question_id` (string): Unique question identifier
- `question_text` (string): The question text
- `retrieval_context` (array): List of documents, each containing a `regions` array
  - `regions` (array): Retrieved chunks with `text` and `chunk_id`

**Field Mapping:**
| Source | Target | Notes |
|--------|--------|-------|
| `question_id` | `question_id` | Direct copy |
| `text` | `question_text` | Rename "text" → "question_text" |
| `context[*]` | `retrieval_context[0].regions[]` | All chunks grouped under a single document |
| `context[i].text` | `regions[i].text` | Direct copy |
| `context[i].chunk_id` | `regions[i].chunk_id` | Direct copy (no rank in Format B) |


In [ ]:
# Define Option B schema template
class RegionItem(TypedDict):
    """A single retrieved chunk (region) in Option B format."""

    text: str
    chunk_id: int | str


class RetrievalDocument(TypedDict):
    """A retrieval document holding a list of regions."""

    regions: list[RegionItem]


class OptionBRecord(TypedDict):
    """A per-question record in Option B format."""

    question_id: str
    question_text: str
    retrieval_context: list[RetrievalDocument]


# Schema validation template
REQUIRED_FIELDS = {
    "question_id": str,
    "question_text": str,
    "retrieval_context": list,
}

RETRIEVAL_DOCUMENT_REQUIRED_FIELDS = {
    "regions": list,
}

REGION_REQUIRED_FIELDS = {
    "text": str,
    "chunk_id": (int, str),
}

print("✓ Option B schema defined")
print(f"  Required top-level fields: {list(REQUIRED_FIELDS.keys())}")
print(
    f"  Required retrieval document fields: {list(RETRIEVAL_DOCUMENT_REQUIRED_FIELDS.keys())}"
)
print(f"  Required region fields: {list(REGION_REQUIRED_FIELDS.keys())}")

✓ Option B schema defined
  Required top-level fields: ['question_id', 'question_text', 'retrieval_context']
  Required retrieval document fields: ['regions']
  Required region fields: ['text', 'chunk_id']


## Section 5: Implement Record Transformation to Option B

In [ ]:
def transform_record(source_record: dict[str, Any]) -> dict[str, Any]:
    """Transform source record to Option B format.

    - question_id: copied as-is
    - question_text: renamed from 'text'
    - retrieval_context: list with a single document whose regions array holds
      the retrieved chunks (text and chunk_id)
    """
    # Build the regions array (no rank field in Format B)
    regions = [
        {
            "text": context_item.get("text"),
            "chunk_id": context_item.get("chunk_id"),
        }
        for context_item in source_record.get("context", [])
    ]

    return {
        "question_id": source_record.get("question_id"),
        "question_text": source_record.get("text"),
        "retrieval_context": [{"regions": regions}],
    }


# Test transformation on first record
if source_data:
    test_transformed = transform_record(source_data[0])
    test_regions = test_transformed["retrieval_context"][0]["regions"]
    print("✓ Transformation function created and tested")
    print("\nFirst record after transformation:")
    print(f"  question_id: {test_transformed['question_id']}")
    print(f"  question_text: {test_transformed['question_text'][:60]}...")
    print(f"  retrieval_context[0].regions: {len(test_regions)} items")
    print("\n  First 2 regions:")
    for i, region in enumerate(test_regions[:2]):
        print(
            f"    [{i}] chunk_id={region['chunk_id']}, text_len={len(region['text'])}"
        )

✓ Transformation function created and tested

First record after transformation:
  question_id: 064ad638-583e-4d7f-941e-597e2e70a9df
  question_text: Which public health controversies shape government actions a...
  retrieval_context[0].regions: 200 items

  First 2 regions:
    [0] chunk_id=5493, text_len=1004
    [1] chunk_id=5492, text_len=1006


## Section 6: Validate Transformed Output

In [ ]:
def validate_record(record: dict[str, Any], _record_idx: int) -> tuple[bool, list[str]]:
    """Validate a transformed record against Option B schema.

    Returns: (is_valid, list of error messages)
    """
    errors = []

    # Check top-level required fields
    if not record.get("question_id"):
        errors.append("Missing or empty question_id")
    if not record.get("question_text"):
        errors.append("Missing or empty question_text")
    if "retrieval_context" not in record:
        errors.append("Missing retrieval_context array")

    # Check retrieval_context structure (list of documents, each with regions)
    retrieval_context = record.get("retrieval_context", [])
    if not isinstance(retrieval_context, list):
        errors.append("retrieval_context is not a list")
    else:
        for doc_idx, doc in enumerate(retrieval_context):
            if not isinstance(doc, dict) or "regions" not in doc:
                errors.append(f"retrieval_context[{doc_idx}] missing regions array")
                continue
            regions = doc.get("regions", [])
            if not isinstance(regions, list):
                errors.append(f"retrieval_context[{doc_idx}].regions is not a list")
                continue
            for region_idx, region in enumerate(regions):
                if not region.get("text"):
                    errors.append(f"doc[{doc_idx}].region[{region_idx}] missing text")
                if region.get("chunk_id") is None:
                    errors.append(
                        f"doc[{doc_idx}].region[{region_idx}] missing chunk_id"
                    )

    return len(errors) == 0, errors


# Transform all records
transformed_data = []
invalid_records = []

if source_data:
    for idx, source_record in enumerate(source_data):
        transformed = transform_record(source_record)
        is_valid, errors = validate_record(transformed, idx)

        if is_valid:
            transformed_data.append(transformed)
        else:
            invalid_records.append({"record_idx": idx, "errors": errors})

    print("✓ Transformation complete")
    print(f"  Total source records: {len(source_data)}")
    print(f"  Valid transformed records: {len(transformed_data)}")
    print(f"  Invalid records: {len(invalid_records)}")

    if invalid_records:
        print("\n⚠ Invalid records:")
        for invalid in invalid_records[:3]:  # Show first 3
            print(f"  Record {invalid['record_idx']}: {invalid['errors']}")

✓ Transformation complete
  Total source records: 50
  Valid transformed records: 50
  Invalid records: 0


## Section 7: Write Option B JSON File

In [ ]:
if transformed_data:
    try:
        with output_file.open("w") as f:
            json.dump(transformed_data, f, indent=2)

        print(
            f"✓ Successfully wrote {len(transformed_data)} records to {output_file.name}"
        )
        print(f"  File size: {output_file.stat().st_size:,} bytes")

    except OSError as e:
        print(f"✗ Error writing output file: {e}")

✓ Successfully wrote 50 records to chunks_b.json
  File size: 10,994,312 bytes


## Section 8: Run Sanity Checks on Saved Output

In [ ]:
if output_file.exists():
    # Reload written file
    with output_file.open() as f:
        verified_data = json.load(f)

    print("=== SANITY CHECKS ===\n")

    # Check record count
    print("Record counts:")
    print(f"  Source: {len(source_data)}")
    print(f"  Transformed: {len(transformed_data)}")
    print(f"  Verified (reloaded): {len(verified_data)}")

    if len(verified_data) == len(source_data):
        print("  ✓ All records preserved")
    else:
        print("  ✗ Record count mismatch!")

    # Sample first record
    print("\nFirst record sample:")
    first = verified_data[0]
    first_regions = first["retrieval_context"][0]["regions"]
    print(f"  question_id: {first['question_id']}")
    print(f"  question_text: {first['question_text'][:70]}...")
    print(f"  retrieval_context[0].regions: {len(first_regions)} items")
    print("\n  First 3 regions:")
    for i, region in enumerate(first_regions[:3]):
        print(
            f"    [{i}] chunk_id={region['chunk_id']}, text_len={len(region['text'])}"
        )

    # Verify structure
    print("\nStructure verification:")
    all_have_retrieval_context = all(
        isinstance(r.get("retrieval_context"), list) for r in verified_data
    )
    all_have_regions = all(
        all("regions" in doc for doc in r.get("retrieval_context", []))
        for r in verified_data
    )

    if all_have_retrieval_context and all_have_regions:
        print("  ✓ All records have retrieval_context (list) with regions structure")
    else:
        print("  ✗ Structure mismatch detected!")

    print("\n✓ Conversion complete and verified!")
else:
    print("Output file not found!")

=== SANITY CHECKS ===

Record counts:
  Source: 50
  Transformed: 50
  Verified (reloaded): 50
  ✓ All records preserved

First record sample:
  question_id: 064ad638-583e-4d7f-941e-597e2e70a9df
  question_text: Which public health controversies shape government actions and legal i...
  retrieval_context[0].regions: 200 items

  First 3 regions:
    [0] chunk_id=5493, text_len=1004
    [1] chunk_id=5492, text_len=1006
    [2] chunk_id=8153, text_len=1143

Structure verification:
  ✓ All records have retrieval_context (list) with regions structure

✓ Conversion complete and verified!
